# Developer: write records through Polaris into SeaweedFS
Run the Markdown setup first. Open this notebook beside private-lab.json in a fresh Python kernel. Run cells in order. This uses isolated run-specific Polaris objects.


In [ ]:
import json, os, socket
from pathlib import Path
from pyspark.sql import SparkSession
config = json.loads(Path('private-lab.json').read_text())
assert config['catalog'].startswith('polaris_ops_')
ROLE = 'developer'
identity = config['principals'][ROLE]
assert SparkSession.getActiveSession() is None, 'Restart the kernel to switch identities safely'
builder = (SparkSession.builder.appName('polaris-' + ROLE + '-' + config['run_id'])
    .master(config['spark_master'])
    .config('spark.driver.host', os.environ.get('POD_IP') or socket.gethostbyname(socket.gethostname()))
    .config('spark.driver.bindAddress', '0.0.0.0')
    .config('spark.cores.max', '1')
    .config('spark.executor.cores', '1')
    .config('spark.executor.memory', '512m')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.redaction.regex', '(?i)secret|password|token|access.?key|credential')
    .config('spark.sql.catalog.lab', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.lab.type', 'rest')
    .config('spark.sql.catalog.lab.uri', config['polaris_uri'])
    .config('spark.sql.catalog.lab.warehouse', config['catalog'])
    .config('spark.sql.catalog.lab.credential', identity['clientId'] + ':' + identity['clientSecret'])
    .config('spark.sql.catalog.lab.scope', 'PRINCIPAL_ROLE:ALL')
    .config('spark.sql.catalog.lab.header.Polaris-Realm', 'POLARIS')
    .config('spark.sql.catalog.lab.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.lab.s3.endpoint', config['s3_endpoint'])
    .config('spark.sql.catalog.lab.s3.path-style-access', 'true')
    .config('spark.sql.catalog.lab.s3.access-key-id', config['s3_access_key'])
    .config('spark.sql.catalog.lab.s3.secret-access-key', config['s3_secret_key'])
    .config('spark.sql.catalog.lab.client.region', 'us-east-1')
    .config('spark.hadoop.fs.s3a.endpoint', config['s3_endpoint'])
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.access.key', config['s3_access_key'])
    .config('spark.hadoop.fs.s3a.secret.key', config['s3_secret_key']))
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.master == config['spark_master']
print('Role:', ROLE, 'Catalog:', config['catalog'], 'Application:', spark.sparkContext.applicationId)
# Never display identity, config or the full SparkConf: they contain credentials.


In [ ]:
spark.sql('SHOW NAMESPACES IN lab').show()
assert spark.table('lab.sales.orders').count() == 0, 'Table already has data: start a fresh exercise run'
orders = spark.createDataFrame([(1,'north',120),(2,'north',80),(3,'south',50),(4,'south',150)],
                               'order_id long, region string, amount long')
orders.writeTo('lab.sales.orders').append()
spark.sql('UPDATE lab.sales.orders SET amount = 100 WHERE order_id = 2')
expected = [(1,'north',120),(2,'north',100),(3,'south',50),(4,'south',150)]
actual = [tuple(r) for r in spark.table('lab.sales.orders').orderBy('order_id').collect()]
assert actual == expected, actual
assert spark.table('lab.sales.orders').groupBy().sum('amount').first()[0] == 420
spark.table('lab.sales.orders').orderBy('order_id').show()
print('PASS: developer inserted four rows, updated one record and read total 420')


In [ ]:
spark.sql('CREATE NAMESPACE lab.audit')
spark.sql("ALTER NAMESPACE lab.audit SET PROPERTIES ('description'='Developer-created namespace')")
spark.sql('CREATE TABLE lab.sales.order_totals USING iceberg AS SELECT region, sum(amount) AS total FROM lab.sales.orders GROUP BY region')
actual_totals = [tuple(r) for r in spark.table('lab.sales.order_totals').orderBy('region').collect()]
assert actual_totals == [('north',220),('south',200)], actual_totals
spark.sql('SHOW TABLES IN lab.sales').show()
print('PASS: developer created another namespace/table and exact regional totals')


In [ ]:
root = f"s3a://{config['bucket']}/polaris-operations/{config['run_id']}/plain-parquet"
orders.write.mode('errorifexists').parquet(root)
assert spark.read.parquet(root).orderBy('order_id').collect() == orders.orderBy('order_id').collect()
location = next(r.data_type for r in spark.sql('DESCRIBE EXTENDED lab.sales.orders').collect() if r.col_name == 'Location')
uri = spark._jvm.java.net.URI(location.replace('s3://','s3a://',1))
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(uri, spark._jsc.hadoopConfiguration())
files = fs.listFiles(spark._jvm.org.apache.hadoop.fs.Path(str(uri)),True)
objects = []
while files.hasNext():
    objects.append(str(files.next().getPath()))
assert any(x.endswith('.parquet') for x in objects)
assert any(x.endswith('.metadata.json') for x in objects)
assert spark.sql('SELECT * FROM lab.sales.orders.snapshots').count() >= 2
print('PASS: independent S3 Parquet round-trip and physical Iceberg data/metadata objects')
print('Table location:', location)


Stop this session before running the viewer notebook. Keep the private config for the same run.


In [ ]:
spark.stop()
print('DEVELOPER_NOTEBOOK_PASS')
